In [1]:
# Create raw order data and a customer master table for end-to-end validation
import pandas as pd
raw_orders = pd.DataFrame({
    "order_id": [1, 2, 2, 3, 4, 5, 6],
    "customer_id": ["C01", "C02", "C02", "C03", "C04", "C99", "C01"],
    "order_date": [
        "2026-01-01",
        "2026-01-05",
        "2026-01-05",
        "invalid",
        "2026-02-10",
        "2026-02-12",
        "2026-02-20",
    ],
    "status": ["complete", "complete", "complete", "cancelled",
               "complete", "complete", "complete"],
    "quantity": [2, 1, 1, 3, -1, 5, 2],
    "unit_price": [20, 50, 50, 15, 100, 10, 25],
})

customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "region": ["North", "South", "North", "West"],
    "segment": ["new", "returning", "new", "business"],
})

In [2]:
# Summarize data-quality issues before cleaning
audit=pd.DataFrame({
    "check":[
        "row_count",
        "duplicate_order_ids",
        "invalid_dates",
        "negative_quantities",
        "unknown_customer_ids",
        "cancelled_orders",
        "missing_values"
    ],
    "count":[
        len(raw_orders),
        raw_orders["order_id"].duplicated().sum(),
        pd.to_datetime(
            raw_orders["order_date"],
            errors="coerce",
        ).isna().sum(),
        (raw_orders["quantity"]<0).sum(),
        (~raw_orders["customer_id"].isin(customers["customer_id"])).sum(),
        (raw_orders["status"]=="cancelled").sum(),
        raw_orders.isna().sum().sum()
    ]
})
audit

,check,count
0,row_count,7
1,duplicate_order_ids,1
2,invalid_dates,1
3,negative_quantities,1
4,unknown_customer_ids,1
5,cancelled_orders,1
6,missing_values,0


In [3]:
# Extract the records responsible for each audit issue
duplicate_orders=raw_orders[
    raw_orders["order_id"].duplicated(keep=False)
]
invalid_date=raw_orders[
    pd.to_datetime(
        raw_orders["order_date"],
        errors="coerce"
    ).isna()
]
negative_quantities=raw_orders[
    raw_orders["quantity"]<0
]
unknown_customers=raw_orders[
    ~raw_orders["customer_id"].isin(customers["customer_id"])
]
cancelled_orders=raw_orders[
    raw_orders["status"]=="cancelled"
]
print("Duplicate Orders:")
print(duplicate_orders)
print("-"*70)
print("Invalid Date:")
print(invalid_date)
print("-"*70)
print("Negative Quantities:")
print(negative_quantities)
print("-"*70)
print("Unknown Customers:")
print(unknown_customers)
print("-"*70)
print("Cancelled Orders:")
print(cancelled_orders)
print("-"*70)

Duplicate Orders:
   order_id customer_id  order_date    status  quantity  unit_price
1         2         C02  2026-01-05  complete         1          50
2         2         C02  2026-01-05  complete         1          50
----------------------------------------------------------------------
Invalid Date:
   order_id customer_id order_date     status  quantity  unit_price
3         3         C03    invalid  cancelled         3          15
----------------------------------------------------------------------
Negative Quantities:
   order_id customer_id  order_date    status  quantity  unit_price
4         4         C04  2026-02-10  complete        -1         100
----------------------------------------------------------------------
Unknown Customers:
   order_id customer_id  order_date    status  quantity  unit_price
5         5         C99  2026-02-12  complete         5          10
----------------------------------------------------------------------
Cancelled Orders:
   order_id cu

In [4]:
# Build an analysis-ready table and retain suspicious records as flags
# Analysis Ready Table
analysis_ready_table=raw_orders.copy()

# safe date parsing
analysis_ready_table["order_date"]=pd.to_datetime(
    analysis_ready_table["order_date"],
    errors="coerce"
)

# remove cancelled orders
analysis_ready_table=analysis_ready_table[analysis_ready_table["status"]!="cancelled"]

# remove duplicate orders
analysis_ready_table=analysis_ready_table.drop_duplicates(
    subset=["order_id"],
    keep="first"
)

# flags for investigation
analysis_ready_table["negative_quantity_flag"]=(
    analysis_ready_table["quantity"]<0
)

analysis_ready_table["unknown_customer_flag"]=(
    ~analysis_ready_table["customer_id"].isin(customers["customer_id"])
)

# revenue column
analysis_ready_table["revenue"]=(
    analysis_ready_table["quantity"]*analysis_ready_table["unit_price"]
)

analysis_ready_table=analysis_ready_table.reset_index(drop=True)
print("ANALYSIS READY TABLE")
print(analysis_ready_table)


ANALYSIS READY TABLE
   order_id customer_id order_date    status  quantity  unit_price  \
0         1         C01 2026-01-01  complete         2          20   
1         2         C02 2026-01-05  complete         1          50   
2         4         C04 2026-02-10  complete        -1         100   
3         5         C99 2026-02-12  complete         5          10   
4         6         C01 2026-02-20  complete         2          25   

   negatice_quantity_flag  unknown_customer_flags  revenue  
0                   False                   False       40  
1                   False                   False       50  
2                    True                   False     -100  
3                   False                    True       50  
4                   False                   False       50  


In [5]:
# Derive day, month, and year features from valid order dates
orders_monthly_features=analysis_ready_table[["order_id","customer_id","order_date"]].copy()
orders_monthly_features["day"]=analysis_ready_table["order_date"].dt.day_name()
orders_monthly_features["month"]=analysis_ready_table["order_date"].dt.month_name()
orders_monthly_features["year"]=analysis_ready_table["order_date"].dt.year
orders_monthly_features=orders_monthly_features.drop(columns="order_date")
orders_monthly_features

,order_id,customer_id,day,month,year
0,1,C01,Thursday,January,2026
1,2,C02,Monday,January,2026
2,4,C04,Tuesday,February,2026
3,5,C99,Thursday,February,2026
4,6,C01,Friday,February,2026


In [6]:
# Prepare records for an audited customer enrichment join
valid_orders=analysis_ready_table.copy()
valid_orders=valid_orders.drop(columns=["negative_quantity_flag","unknown_customer_flag","revenue"])
valid_orders

,order_id,customer_id,order_date,status,quantity,unit_price
0,1,C01,2026-01-01,complete,2,20
1,2,C02,2026-01-05,complete,1,50
2,4,C04,2026-02-10,complete,-1,100
3,5,C99,2026-02-12,complete,5,10
4,6,C01,2026-02-20,complete,2,25


In [7]:
# Join customer attributes while enforcing a many-to-one relationship
order_customer=valid_orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)
order_customer

,order_id,customer_id,order_date,status,quantity,unit_price,region,segment
0,1,C01,2026-01-01,complete,2,20,North,new
1,2,C02,2026-01-05,complete,1,50,South,returning
2,4,C04,2026-02-10,complete,-1,100,West,business
3,5,C99,2026-02-12,complete,5,10,NaN,NaN
4,6,C01,2026-02-20,complete,2,25,North,new


In [8]:
# Calculate revenue and keep negative values out of the reported total
order_customer["revenue"]=(order_customer["quantity"]*order_customer["unit_price"]).clip(lower=0)
order_customer

,order_id,customer_id,order_date,status,quantity,unit_price,region,segment,revenue
0,1,C01,2026-01-01,complete,2,20,North,new,40
1,2,C02,2026-01-05,complete,1,50,South,returning,50
2,4,C04,2026-02-10,complete,-1,100,West,business,0
3,5,C99,2026-02-12,complete,5,10,NaN,NaN,50
4,6,C01,2026-02-20,complete,2,25,North,new,50


In [11]:
# Preserve excluded and suspicious records in an audit table
# Records excluded from analysis
excluded_records=raw_orders[
    (raw_orders["status"]=="cancelled")
    |
    (raw_orders["order_id"].duplicated(keep="first"))
].copy()

excluded_records["audit_reason"]=""

excluded_records.loc[
    excluded_records["status"]=="cancelled",
    "audit_reason"
]+="cancelled_order "

excluded_records.loc[
    raw_orders["order_id"].duplicated(keep="first"),
    "audit_reason"
]+="duplicated_order_id "

# Suspicious records kept for investigation
suspicious_records=raw_orders[
    (raw_orders["quantity"]<0)
    |
    (~raw_orders["customer_id"].isin(customers["customer_id"]))
].copy()

suspicious_records["audit_reason"]=""
suspicious_records.loc[
    suspicious_records["quantity"]<0,"audit_reason"
]+="negative_quantity "
suspicious_records.loc[
    ~suspicious_records["customer_id"].isin(customers["customer_id"]),
    "audit_reason"
]+="unknown_customer "

# Combined audit table
audit_records=pd.concat(
    [excluded_records,suspicious_records],
    ignore_index=True
)
audit_records



,order_id,customer_id,order_date,status,quantity,unit_price,audit_reason
0,2,C02,2026-01-05,complete,1,50,duplicated_order_id
1,3,C03,invalid,cancelled,3,15,cancelled_order
2,4,C04,2026-02-10,complete,-1,100,negative_quantity
3,5,C99,2026-02-12,complete,5,10,unknown_customer


### Conclusion

The dataset was cleaned by removing cancelled orders and duplicate order records. However, certain records were retained and flagged for investigation, including orders with negative quantities and unknown customer IDs.

As a result, the cleaned dataset is more suitable for analysis, but the findings should be interpreted with caution because unresolved data-quality issues remain. Revenue calculations and customer-level analysis may still be affected if the flagged records are later found to be incorrect.


### Important notes

- Run audit checks before cleaning so data-quality issues remain visible and measurable.
- Invalid dates become `NaT` with `pd.to_datetime(errors="coerce")`; inspect them before deriving calendar features.
- Duplicate order IDs should be resolved according to source-system rules, not removed blindly; the exercise keeps the first record for a reproducible example.
- Suspicious records can be flagged and retained for investigation instead of silently changing or deleting their values.
- A left join preserves all valid orders, while `validate="many_to_one"` ensures each customer key is unique on the right table.
- `clip(lower=0)` prevents negative quantity revenue; document the final decision for negative quantities before publishing totals.
- Reconcile audit counts, row counts, and revenue totals after cleaning and joining to catch accidental row loss or multiplication.
